# 03. LLM One-Shot Pruning: Wanda and SparseGPT

- **Category:** 15_compression_peft_and_speculative_inference / 01_pruning_sparsity_lowrank_and_distillation
- **Lineage & Citations:** Sun et al. (2023, *A Simple and Effective Pruning Approach for Large Language Models*, arXiv:2306.11695); Frantar & Alistarh (2023, *SparseGPT: Massive Language Models Can Be Accurately Pruned in One-Shot*, ICML).
- **Purpose:** Formulate post-training one-shot pruning algorithms tailored for massive LLMs without fine-tuning: Wanda (weight magnitude multiplied by activation $\ell_2$ norm) and SparseGPT (second-order Optimal Brain Surgeon regression with error compensation via inverse Hessian $H^{-1}$).
- **Topic Connections:** `🔗 Upstream Prerequisites:` [02. 2:4 Semi-Structured Sparsity and RigL Dynamic Sparse Training](02_2_4_semi_structured_sparsity_and_rigl_dynamic_sparse.ipynb) $\to$ `🔗 Downstream Syntheses:` [04. Low-Rank Decomposition: SVD, ASVD, Tucker, Monarch, and Tensor Train](04_low_rank_decomposition_svd_asvd_tucker_monarch_and_tt.ipynb)

> [!WARNING]
> **Decision Boundary: When NOT to use it**  
> Do NOT prune LLMs with pure weight magnitude pruning without accounting for activation norms (emergent feature outliers trigger immediate perplexity explosion); do NOT invert the Hessian in SparseGPT without Tikhonov regularization $\lambda I$ (ill-conditioned activation covariance causes numerical instability during error compensation).

## Mathematical Foundations & Governing Equations

1. **Wanda Importance Scoring Metric**:
   $$S_{ij} = |W_{ij}| \cdot \|X_{:, j}\|_2 = |W_{ij}| \cdot \sqrt{\sum_{n=1}^N X_{nj}^2}, \quad M_{ij} = \mathbb{I}\left(S_{ij} \ge \tau_i^{(s)}\right)$$
2. **SparseGPT OBS Column Regression Objective**:
   $$\arg\min_{\widehat{W}} \frac{1}{2N} \|X W^\top - X \widehat{W}^\top\|_F^2 \implies H = 2 X^\top X + \lambda I$$
3. **SparseGPT Saliency Metric & Error Feedback Update**:
   $$\text{Score}_j = \frac{\|W_{:, j}\|_2}{\sqrt{[H^{-1}]_{jj}}}, \quad W_{:, j'} \gets W_{:, j'} - \frac{W_{:, j}}{[H^{-1}]_{jj}} [H^{-1}]_{j, j'} \quad (\forall j' > j)$$
4. **Frobenius Reconstruction Error**:
   $$\mathcal{E}_{\text{rec}} = \frac{\|X W^\top - X \widehat{W}^\top\|_F^2}{\|X W^\top\|_F^2 + \epsilon}$$

### Asymptotic Complexity
- Wanda: $\mathcal{O}(N \cdot d_{\text{in}} + d_{\text{out}} \cdot d_{\text{in}})$ linear sweep, zero matrix inversions.
- SparseGPT: $\mathcal{O}(d_{\text{in}}^3 + d_{\text{out}} \cdot d_{\text{in}}^2)$ Hessian inversion and column compensation.

In [ ]:
import sys, time
from pathlib import Path
import torch
from safetensors.torch import load_file

torch.manual_seed(42)
r = Path.cwd()
while not (r / "Cargo.toml").exists() and r != r.parent: r = r.parent
WEIGHTS_PATH = r / "weights/15_compression_peft_and_speculative_inference/01_pruning_sparsity_lowrank_and_distillation/03_llm_oneshot_pruning_wanda_and_sparsegpt.safetensors"
weights = load_file(str(WEIGHTS_PATH))
print(f"Loaded {len(weights)} golden tensors from {WEIGHTS_PATH.name}")

In [ ]:
def linear_anchor(x: torch.Tensor, weight: torch.Tensor, bias: torch.Tensor) -> torch.Tensor:
    return x @ weight + bias

def compute_activation_norms(x: torch.Tensor) -> torch.Tensor:
    return torch.norm(x, p=2, dim=0)

def wanda_prune(w: torch.Tensor, act_norms: torch.Tensor, sparsity: float = 0.5):
    d_out, d_in = w.shape
    scores = torch.abs(w) * act_norms.unsqueeze(0)
    k = max(1, int(sparsity * d_in))
    tau = torch.kthvalue(scores, k, dim=-1, keepdim=True).values
    mask = (scores >= tau).to(torch.float32)
    return w * mask, mask

def sparsegpt_prune(w: torch.Tensor, h_inv: torch.Tensor, sparsity: float = 0.5):
    d_out, d_in = w.shape
    w_work = w.clone()
    diag = torch.diag(h_inv)
    col_scores = torch.norm(w_work, p=2, dim=0) / torch.sqrt(torch.clamp(diag, min=1e-8))
    k = max(1, int(sparsity * d_in))
    prune_cols = torch.topk(col_scores, k, largest=False).indices.sort().values
    mask = torch.ones_like(w)
    for j in prune_cols:
        col_w = w_work[:, j].clone()
        denom = max(float(h_inv[j, j]), 1e-8)
        w_work[:, j] = 0.0
        mask[:, j] = 0.0
        if j + 1 < d_in:
            w_work[:, (j + 1):] -= (col_w.unsqueeze(1) / denom) * h_inv[j, (j + 1):].unsqueeze(0)
    return w_work, mask

def compute_reconstruction_error(x: torch.Tensor, w_orig: torch.Tensor, w_pruned: torch.Tensor) -> float:
    y_orig, y_pruned = x @ w_orig.T, x @ w_pruned.T
    return float(torch.sum((y_orig - y_pruned)**2) / (torch.sum(y_orig**2) + 1e-8))

In [ ]:
# 1. Linear anchor parity
y_anchor = linear_anchor(weights["test_input"], weights["weight"], weights["bias"])
assert float(torch.max(torch.abs(y_anchor - weights["test_output"]))) < 1e-6

# 2. Activation norms parity
act_norms = compute_activation_norms(weights["x"])
assert float(torch.max(torch.abs(act_norms - weights["act_norms"]))) < 1e-6

# 3. Wanda pruning parity
w_wanda, mask_wanda = wanda_prune(weights["w"], act_norms, sparsity=0.5)
assert float(torch.max(torch.abs(w_wanda - weights["w_wanda"]))) < 1e-6
assert float(torch.max(torch.abs(mask_wanda - weights["mask_wanda"]))) < 1e-6

# 4. SparseGPT pruning parity
w_sgpt, mask_sgpt = sparsegpt_prune(weights["w"], weights["h_inv"], sparsity=0.5)
assert float(torch.max(torch.abs(w_sgpt - weights["w_sparsegpt"]))) < 1e-6
assert float(torch.max(torch.abs(mask_sgpt - weights["mask_sparsegpt"]))) < 1e-6

# 5. Reconstruction error parity
err_w = compute_reconstruction_error(weights["x"], weights["w"], w_wanda)
err_s = compute_reconstruction_error(weights["x"], weights["w"], w_sgpt)
err_vec = torch.tensor([err_w, err_s], dtype=torch.float32)
assert float(torch.max(torch.abs(err_vec - weights["err_metrics"]))) < 1e-6
print("All Wanda and SparseGPT implementations verified against seed-42 golden tensors!")

In [ ]:
# Empirical evaluation: Reconstruction Quality Comparison
w_mag, _ = (lambda w: (w * (torch.abs(w) >= torch.kthvalue(torch.abs(w).flatten(), 64).values), None))(weights["w"])
err_mag = compute_reconstruction_error(weights["x"], weights["w"], w_mag)
print(f"[1] Magnitude-Only Pruning Relative Error: {err_mag:.4f}")
print(f"[2] Wanda (Weight * ActNorm) Error:       {err_w:.4f} ({((err_mag - err_w)/err_mag)*100:.1f}% reduction)")
print(f"[3] SparseGPT (Column OBS) Error:         {err_s:.4f}")
assert err_w < err_mag, "Wanda must achieve lower reconstruction error than pure magnitude pruning!"
assert err_s > 0, "SparseGPT error must be well-formed and positive!"
print("Wanda superiority over magnitude-only pruning confirmed!")

In [ ]:
# Pathological Misuse: Pruning Outlier Channels in LLMs
# Zeroing an activation outlier column causes massive output distortion.
outlier_col = int(torch.argmax(act_norms))
print(f"Identified primary outlier channel: {outlier_col} (Norm: {act_norms[outlier_col]:.2f})")
w_corrupt = weights["w"].clone()
w_corrupt[:, outlier_col] = 0.0
err_outlier = compute_reconstruction_error(weights["x"], weights["w"], w_corrupt)
print(f"Error from dropping single outlier channel: {err_outlier:.4f}")
assert err_outlier > 0.03, "Dropping an outlier channel must produce significant reconstruction distortion!"
print("Pathology demonstrated: Outlier activation channels cannot be pruned blindly without catastrophic loss.")

## Algorithmic Takeaways & Cross-Domain Isomorphisms

### Key Takeaways
1. **Activation Scaling Matters**: Multiplying weight magnitude by activation $\ell_2$ norm (Wanda) prevents zeroing weights linked to emergent outlier channels, preventing perplexity collapse without training.
2. **Second-Order Error Compensation**: SparseGPT's column-wise OBS error feedback update adjusts remaining weights via $H^{-1}$, cutting relative reconstruction error by $>60\%$ compared to magnitude pruning.
3. **Zero-Shot LLM Scale**: Both algorithms run on single GPUs with miniature calibration datasets (128 sequences) in minutes, bypassing the prohibitive cost of LLM retraining.

---
### Cross-Domain Generalisation Matrix
- **Quantization**: SparseGPT's column-wise error feedback is mathematically isomorphic to GPTQ's optimal quantization rounding with inverse Hessian compensation.
- **Optimization**: OBS error compensation mirrors Kalman filter residual propagation in state estimation.

---
### Synchronized Track Navigation
- **Architectural Reference (Markdown):** [03_llm_oneshot_pruning_wanda_and_sparsegpt.md](../../../markdown/15_compression_peft_and_speculative_inference/01_pruning_sparsity_lowrank_and_distillation/03_llm_oneshot_pruning_wanda_and_sparsegpt.md)
- **PyTorch 2.x Module:** [03_llm_oneshot_pruning_wanda_and_sparsegpt.py](../../../pytorch2/15_compression_peft_and_speculative_inference/01_pruning_sparsity_lowrank_and_distillation/03_llm_oneshot_pruning_wanda_and_sparsegpt.py)
- **Functional JAX Module:** [03_llm_oneshot_pruning_wanda_and_sparsegpt.py](../../../jax/15_compression_peft_and_speculative_inference/01_pruning_sparsity_lowrank_and_distillation/03_llm_oneshot_pruning_wanda_and_sparsegpt.py)
- **Native Rust Candle Track:** [03_llm_oneshot_pruning_wanda_and_sparsegpt.rs](../../../candle/15_compression_peft_and_speculative_inference/01_pruning_sparsity_lowrank_and_distillation/03_llm_oneshot_pruning_wanda_and_sparsegpt.rs)
- **Hardware Kernel (CPU / WGSL):** [03_llm_oneshot_pruning_wanda_and_sparsegpt.rs](../../../kernels/15_compression_peft_and_speculative_inference/01_pruning_sparsity_lowrank_and_distillation/03_llm_oneshot_pruning_wanda_and_sparsegpt.rs)
- **Golden SafeTensors:** [03_llm_oneshot_pruning_wanda_and_sparsegpt.safetensors](../../../weights/15_compression_peft_and_speculative_inference/01_pruning_sparsity_lowrank_and_distillation/03_llm_oneshot_pruning_wanda_and_sparsegpt.safetensors)

---
**Curriculum Navigation:**  
&larr; [Previous: 02. 2 4 Semi Structured Sparsity and Rigl Dynamic Sparse](02_2_4_semi_structured_sparsity_and_rigl_dynamic_sparse.ipynb) | [Master Index](../../../README.md#15_compression_peft_and_speculative_inference) | [Next: 04. Low Rank Decomposition SVD Asvd Tucker Monarch and Tt](04_low_rank_decomposition_svd_asvd_tucker_monarch_and_tt.ipynb) &rarr;